# MVP Análise de Dados e Boas Práticas

**Nome:** Raquel Paradella Freitas Maranhão

**Matrícula:** 4052026000590

**Dataset:** [Smartphone Usage And Addiction Analysis](https://www.kaggle.com/datasets/zahranusratt/smartphone-usage-and-addiction-analysis-dataset)

# Descrição do Problema

O conjunto de dados **Smartphone Usage And Addiction Analysis** reúne registros comportamentais de 7.500 usuários com idades entre 18 e 35 anos. O objetivo principal é analisar padrões de uso do smartphone e identificar quais comportamentos estão associados ao vício digital (`addiction_level`) e ao rótulo binário de dependência (`addicted_label`).

O problema é relevante porque o uso excessivo de smartphones tem sido associado a impactos negativos na saúde mental, qualidade do sono e desempenho acadêmico/profissional.

## Hipóteses do Problema

As hipóteses que tracei são as seguintes:

1. **Usuários com maior `daily_screen_time_hours` tendem a apresentar `addiction_level` Severe e `addicted_label = 1`?**

2. **Existe correlação entre `daily_screen_time_hours` e `weekend_screen_time`?**

3. **Usuários com `stress_level = High` utilizam mais o smartphone do que usuários com `stress_level = Low`?**

## Tipo de Problema

Este é um problema de **classificação supervisionada**. Dado um conjunto de características comportamentais de uso do smartphone (tempo de tela, horas em redes sociais, jogos, notificações etc.), o objetivo é prever se o usuário é dependente (`addicted_label = 1`) ou não (`addicted_label = 0`), além de estimar o nível de dependência (`addiction_level`: Mild, Moderate ou Severe).

## Seleção de Dados

O dataset foi obtido a partir de uma coleta estruturada de dados comportamentais de usuários de smartphones. Cada linha representa um registro de usuário com métricas de uso diário e indicadores de bem-estar. O arquivo está disponível no formato CSV e foi carregado diretamente para análise. A variável `addiction_level` possui **819 valores ausentes (~10,9%)**, o que será investigado durante a EDA.

## Atributos do Dataset

O dataset contém 7.500 instâncias (observações) e 16 atributos:

- **`transaction_id`** / **`user_id`**: identificadores únicos de registro
- **`age`** (int): idade do usuário (18-35 anos)
- **`gender`** (categórico): gênero - Male, Female, Other
- **`daily_screen_time_hours`** (float): tempo total de tela diário (horas)
- **`social_media_hours`** (float): horas diárias em redes sociais
- **`gaming_hours`** (float): horas diárias em jogos
- **`work_study_hours`** (float): horas diárias em trabalho/estudo
- **`sleep_hours`** (float): horas de sono por noite
- **`notifications_per_day`** (int): notificações recebidas por dia
- **`app_opens_per_day`** (int): aberturas de apps por dia
- **`weekend_screen_time`** (float): tempo de tela nos fins de semana (horas)
- **`stress_level`** (categórico ordinal): nível de estresse - Low, Medium, High
- **`academic_work_impact`** (binário): impacto negativo no desempenho - Yes / No
- **`addiction_level`** (categórico ordinal): grau de vício - Mild, Moderate, Severe (819 nulos)
- **`addicted_label`** (binário 0/1): rótulo de classificação - 1 = viciado, 0 = não viciado

# Importação das Bibliotecas Necessárias e Carga de Dados

Esta seção consolida todas as importações de bibliotecas necessárias para a análise, visualização e pré-processamento dos dados, bem como o carregamento inicial do dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# carregamento do dataset

# URL da versão raw do dataset no GitHub
url = 'https://raw.githubusercontent.com/raquelpfm/MVP-001/refs/heads/main/Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv'

# Carregamento do dataset a partir da URL
df = pd.read_csv(url)

df.head()

In [ ]:
# Removendo colunas de identificação que não são features
df_clean = df.drop(columns=['transaction_id', 'user_id'])

In [ ]:
# primeiras linhas
df_clean.head()

# Análise de Dados

Nesta etapa de Análise de Dados Exploratória (EDA) sobre o dataset de uso de smartphones, visamos entender a distribuição, as relações e as características das variáveis, o que é crucial para as etapas subsequentes de pré-processamento e modelagem.

## Total e Tipo das Instâncias

O dataset possui **7.500 instâncias** e **16 atributos**. Os atributos numéricos incluem medições contínuas de uso (horas de tela, sono, notificações etc.) e os atributos categóricos representam características de perfil e de impacto comportamental.

In [ ]:
print(f"Total de instâncias: {len(df_clean)}")
print("\nTipos de dados por coluna:")
print(df_clean.info())

In [ ]:
plt.figure(figsize=(7, 5))
# gráfico de barras: distribuição da variável alvo
sns.countplot(x='addicted_label', data=df_clean)
plt.title('Distribuição da Variável Alvo (addicted_label)')
plt.xlabel('Addicted Label (0 = Não, 1 = Sim)')
plt.ylabel('Contagem')
plt.xticks([0, 1], ['Não Viciado (0)', 'Viciado (1)'])
plt.show()

O gráfico de barras mostra que a classe majoritária é `addicted_label = 1` (viciado), representando aproximadamente **70,8%** dos registros. O dataset é **desbalanceado**, o que pode demandar estratégias como oversampling (SMOTE) ou ajuste de peso de classes na etapa de modelagem.

## Estatísticas Descritivas

Estatísticas descritivas fornecem um resumo das características numéricas, incluindo média, desvio padrão, mínimo, máximo e quartis.

In [ ]:
# estatísticas descritivas básicas do dataset
df_clean.describe()

### Média

A média é uma medida de tendência central que representa o valor típico ou o ponto de equilíbrio de um conjunto de dados. É calculada somando-se todos os valores e dividindo-se pelo número total de observações. É sensível a valores extremos (outliers).

In [ ]:
# média dos atributos numéricos do dataset
num_cols = df_clean.select_dtypes(include='number').drop(columns=['addicted_label']).columns
df_clean[num_cols].describe().loc['mean']

In [ ]:
# Gráfico de barras da média das principais variáveis numéricas
medias = df_clean[num_cols].mean()
plt.figure(figsize=(10, 5))
medias.plot(kind='bar', color='steelblue')
plt.title('Média das Variáveis Numéricas')
plt.ylabel('Valor médio')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Desvio Padrão

O desvio padrão é uma medida de dispersão que quantifica a quantidade de variação ou dispersão de um conjunto de valores. Um desvio padrão baixo indica que os pontos de dados tendem a estar próximos da média do conjunto, enquanto um desvio padrão alto indica que os pontos de dados estão espalhados por uma faixa maior de valores. Ele é a raiz quadrada da variância.

In [ ]:
# desvio padrão dos atributos numéricos do dataset
df_clean[num_cols].describe().loc['std']

In [ ]:
# Boxplot para visualizar dispersão das principais variáveis
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ['daily_screen_time_hours', 'sleep_hours', 'social_media_hours']):
    sns.boxplot(y=df_clean[col], ax=ax)
    ax.set_title(col)
plt.suptitle('Dispersão das Variáveis de Uso')
plt.tight_layout()
plt.show()

## Histograma

A distribuição de dados descreve como os valores de uma variável se espalham, ou seja, a frequência com que diferentes valores ocorrem. Entender a distribuição é crucial na análise de dados, pois revela padrões, tendências centrais, dispersão e a presença de valores atípicos (outliers). O histograma é uma ferramenta visual fundamental para representar essa distribuição.

### *daily_screen_time_hours*

Esta variável representa o tempo total de tela diário do usuário. Esperamos uma distribuição com cauda longa à direita, indicando a presença de usuários com uso muito elevado.

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df_clean['daily_screen_time_hours'], kde=True)
plt.title('Distribuição do Tempo de Tela Diário')
plt.xlabel('Tempo de Tela Diário (horas)')
plt.ylabel('Frequência')
plt.show()

O histograma do `daily_screen_time_hours` mostra uma distribuição **aproximadamente uniforme** entre 3 e 12 horas, com leve tendência central. A curva KDE evidencia que a maioria dos usuários utiliza o smartphone entre 5 e 10 horas por dia.

### *sleep_hours*

A quantidade de horas de sono é um indicador importante de bem-estar. Usuários com padrões de uso intensivo tendem a dormir menos.

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df_clean['sleep_hours'], kde=True)
plt.title('Distribuição de Horas de Sono')
plt.xlabel('Horas de Sono por Noite')
plt.ylabel('Frequência')
plt.show()

A distribuição de `sleep_hours` é relativamente simétrica, centrada em torno de 6–7 horas. Isso sugere que, embora haja variação, a maioria dos usuários dorme dentro de uma faixa considerada normal.

## Boxplot

Para entender as diferenças entre grupos, devemos observar como as variáveis se comportam quando agrupadas por categorias como `addiction_level` e `stress_level`.

In [ ]:
# Estatísticas descritivas agrupadas por nível de vício
df_clean.groupby('addiction_level')[['daily_screen_time_hours', 'sleep_hours', 'social_media_hours']].describe()

### *daily_screen_time_hours* por Nível de Vício

In [ ]:
plt.figure(figsize=(10, 6))
# Ordenar por nível de gravidade
order = ['Mild', 'Moderate', 'Severe']
df_nonnull = df_clean.dropna(subset=['addiction_level'])
sns.boxplot(x='addiction_level', y='daily_screen_time_hours', data=df_nonnull, order=order)
plt.title('Tempo de Tela Diário por Nível de Vício')
plt.xlabel('Nível de Vício')
plt.ylabel('Tempo de Tela Diário (horas)')
plt.show()

O boxplot revela que usuários com `addiction_level = Severe` apresentam maior mediana de `daily_screen_time_hours` em comparação com Mild e Moderate, o que é coerente com a hipótese de que maior tempo de tela está associado a maior nível de dependência.

### *sleep_hours* por Nível de Estresse

In [ ]:
plt.figure(figsize=(10, 6))
order_stress = ['Low', 'Medium', 'High']
sns.boxplot(x='stress_level', y='sleep_hours', data=df_clean, order=order_stress)
plt.title('Horas de Sono por Nível de Estresse')
plt.xlabel('Nível de Estresse')
plt.ylabel('Horas de Sono')
plt.show()

O boxplot de `sleep_hours` por `stress_level` mostra que usuários com estresse `High` tendem a dormir **ligeiramente menos** do que usuários com estresse `Low`, sugerindo uma relação entre alto uso de smartphone, estresse e privação de sono.

## Matriz de Correlação

A matriz de correlação mede a força e a direção de uma relação linear entre atributos numéricos. Valores próximos a 1 indicam forte correlação positiva, -1 forte correlação negativa, e 0 ausência de correlação linear.

In [ ]:
# Matriz de correlação
print("\nMatriz de Correlação:")
df_clean[num_cols].corr()

In [ ]:
plt.figure(figsize=(10, 8))
# mapa de calor das variáveis numéricas
sns.heatmap(df_clean[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação das Variáveis Numéricas')
plt.tight_layout()
plt.show()

O mapa de calor da matriz de correlação revela uma **fortíssima correlação positiva** entre `daily_screen_time_hours` e `weekend_screen_time` (~0.96), indicando que o comportamento de uso é consistente entre dias úteis e fins de semana. As demais variáveis apresentam correlações muito baixas entre si, sugerindo **independência estatística** entre a maioria dos atributos comportamentais.

## Tratamento de Valores Nulos

A variável `addiction_level` possui **819 valores ausentes (~10,9%)**. Isso deve ser investigado, pode indicar ausência de avaliação clínica para parte da amostra. Para a variável-alvo binária (`addicted_label`), não há valores ausentes.

In [ ]:
# Verificar a presença de valores nulos no dataset
print("Valores nulos por coluna:")
print(df_clean.isnull().sum())

# Pré-Processamento de Dados

O pré-processamento de dados é uma etapa crucial para preparar os dados para modelagem, garantindo que estejam no formato correto e otimizados para o desempenho do algoritmo.

In [ ]:
# Separar features (X) e target (y)
# Remover colunas não-features e tratar valores nulos de addiction_level
df_model = df_clean.copy()

# Preencher nulos de addiction_level com a moda
df_model['addiction_level'] = df_model['addiction_level'].fillna(df_model['addiction_level'].mode()[0])

# Codificar variáveis categóricas ordinais
stress_map = {'Low': 0, 'Medium': 1, 'High': 2}
addiction_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}
df_model['stress_level'] = df_model['stress_level'].map(stress_map)
df_model['addiction_level'] = df_model['addiction_level'].map(addiction_map)
df_model['academic_work_impact'] = (df_model['academic_work_impact'] == 'Yes').astype(int)
df_model['gender'] = LabelEncoder().fit_transform(df_model['gender'])

X = df_model.drop(columns=['addicted_label'])
y = df_model['addicted_label']

In [ ]:
# Dividir os dados em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
print(f"Dimensões de X_train: {X_train.shape}")
print(f"Dimensões de X_test: {X_test.shape}")
print(f"Dimensões de y_train: {y_train.shape}")
print(f"Dimensões de y_test: {y_test.shape}")

## Normalização

A normalização escala os dados para um intervalo fixo, geralmente entre 0 e 1. É útil quando o algoritmo de machine learning assume que as características estão em uma escala semelhante.

In [ ]:
# Inicializar o MinMaxScaler
scaler_norm = MinMaxScaler()

In [ ]:
# Aprende min e max APENAS de X_train
scaler_norm.fit(X_train)
X_train_normalized = scaler_norm.transform(X_train)
# Usa os parâmetros aprendidos de X_train para transformar X_test
X_test_normalized = scaler_norm.transform(X_test)

In [ ]:
# Exibir as primeiras linhas dos dados normalizados (como DataFrame para melhor visualização)
df_normalized = pd.DataFrame(X_train_normalized, columns=X_train.columns)

In [ ]:
print("\nPrimeiras 5 linhas dos dados normalizados (treino):")
print(df_normalized.head())

In [ ]:
# Visualização da distribuição após a normalização (exemplo para uma característica)
plt.figure(figsize=(8, 6))
sns.histplot(df_normalized['daily_screen_time_hours'], kde=True)
plt.title('Distribuição do Tempo de Tela Diário (Normalizado)')
plt.xlabel('Tempo de Tela Normalizado')
plt.ylabel('Frequência')
plt.show()

O histograma de `daily_screen_time_hours` após a normalização mostra que os valores foram escalados para o intervalo de 0 a 1, **mantendo a forma da distribuição original** (aproximadamente uniforme).

## Padronização

A padronização (ou Z-score scaling) transforma os dados para ter média 0 e desvio padrão 1. É útil para algoritmos sensíveis à escala das características, como SVMs ou redes neurais.

In [ ]:
# Inicializar o StandardScaler
scaler_std = StandardScaler()

In [ ]:
# Aprende média e desvio padrão APENAS de X_train
scaler_std.fit(X_train)
X_train_standardized = scaler_std.transform(X_train)
# Usa a média e o desvio padrão aprendidos de X_train
X_test_standardized = scaler_std.transform(X_test)

In [ ]:
# Exibir as primeiras linhas dos dados padronizados (como DataFrame para melhor visualização)
df_standardized = pd.DataFrame(X_train_standardized, columns=X_train.columns)

In [ ]:
print("\nPrimeiras 5 linhas dos dados padronizados (treino):")
print(df_standardized.head())

In [ ]:
# Visualização da distribuição após a padronização (exemplo para uma característica)
plt.figure(figsize=(8, 6))
sns.histplot(df_standardized['daily_screen_time_hours'], kde=True)
plt.title('Distribuição do Tempo de Tela Diário (Padronizado)')
plt.xlabel('Tempo de Tela Padronizado (Z-score)')
plt.ylabel('Frequência')
plt.show()

O histograma de `daily_screen_time_hours` após a padronização mostra que os valores foram transformados para ter uma **média próxima de zero** e **desvio padrão de um**, centralizando a distribuição.

## Outras Transformações e Etapas de Pré-Processamento

Outras etapas de pré-processamento incluem a seleção de características, redução de dimensionalidade (como PCA) ou criação de novas características (feature engineering). O que você faria a mais?

Por exemplo, para este dataset, seria interessante:
- Criar uma variável `lazer_ratio = (social_media_hours + gaming_hours) / daily_screen_time_hours` para capturar a proporção de uso não produtivo.
- Investigar se o padrão de dados ausentes em `addiction_level` é sistemático (MCAR, MAR ou MNAR).

# Respondendo nossas hipóteses

Quais visualizações, tabelas e células descritivas respondem às hipóteses que levantamos?

## Hipótese 1

In [ ]:
# Hipótese 1: usuários com maior daily_screen_time_hours tendem a ser classificados como Severe?
# Comparando a distribuição de screen time por addiction_level
df_nonnull = df_clean.dropna(subset=['addiction_level'])
print("Média de daily_screen_time_hours por addiction_level:")
print(df_nonnull.groupby('addiction_level')['daily_screen_time_hours'].mean().sort_values(ascending=False))

## Hipótese 2

In [ ]:
# Hipótese 2: correlação entre daily_screen_time_hours e weekend_screen_time
corr_val = df_clean['daily_screen_time_hours'].corr(df_clean['weekend_screen_time'])
print(f"Correlação de Pearson entre daily_screen_time_hours e weekend_screen_time: {corr_val:.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(df_clean['daily_screen_time_hours'], df_clean['weekend_screen_time'], alpha=0.3, s=5)
plt.xlabel('Tempo de Tela Diário (horas)')
plt.ylabel('Tempo de Tela no Fim de Semana (horas)')
plt.title('Relação entre Tempo de Tela Diário e de Fim de Semana')
plt.tight_layout()
plt.show()

## Hipótese 3

In [ ]:
# Hipótese 3: usuários com stress_level=High usam mais o smartphone?
print("Média de daily_screen_time_hours por stress_level:")
print(df_clean.groupby('stress_level')['daily_screen_time_hours'].mean().sort_values(ascending=False))

plt.figure(figsize=(8, 5))
order_stress = ['Low', 'Medium', 'High']
sns.barplot(x='stress_level', y='daily_screen_time_hours', data=df_clean, order=order_stress, ci=95)
plt.title('Tempo de Tela Médio por Nível de Estresse')
plt.xlabel('Nível de Estresse')
plt.ylabel('Tempo de Tela Médio (horas)')
plt.show()

# Conclusão

A análise e pré-processamento do dataset de uso de smartphones demonstram a importância de entender a estrutura dos dados antes da modelagem. O dataset possui variáveis comportamentais bem definidas que permitem investigar padrões de dependência digital.

A análise exploratória revelou insights importantes:

1. **Hipótese 1 - Tempo de tela e nível de vício:** Os resultados confirmam que usuários com `addiction_level = Severe` apresentam maior média de `daily_screen_time_hours`, validando a hipótese.

2. **Hipótese 2 - Correlação entre uso diário e no fim de semana:** Existe uma correlação muito forte (~0.96) entre `daily_screen_time_hours` e `weekend_screen_time`, indicando comportamento consistente de uso ao longo da semana, o que valida a hipótese.

3. **Hipótese 3 - Estresse e uso do smartphone:** Os dados mostram diferença relativamente pequena no tempo de tela médio entre os grupos de estresse, sugerindo que a hipótese precisa de investigação mais aprofundada ou que outras variáveis mediam essa relação.

As etapas de normalização e padronização foram aplicadas corretamente, o modelo é ajustado **somente nos dados de treino** e aplicado nos dados de teste, evitando *data leakage*.